In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/reto.parquet")

# 1) Normalizar booleans (todo el mundo es "object", toca bajarlo a bool)
def to_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "t", "yes"])

df["isComment_bool"] = to_bool(df["isComment"])
df["isRetweet_bool"] = to_bool(df["isRetweet"])
df["isDeleted_bool"] = to_bool(df["isDeleted"])
df["isAdvertisement_bool"] = to_bool(df["isAdvertisement"])
df["isBot_bool"] = to_bool(df["isBot"])

# 2) Limpiar parentId: marcar como NaN los "no padre"
no_parent_tokens = ["", "null", "none", "nan", "na", "nil", "0"]

df["parentId_clean"] = (
    df["parentId"]
    .astype(str)
    .str.strip()
    .replace(no_parent_tokens, np.nan)
)

# 3) Candidatos a raíz a nivel global
root_candidates = df[
    (~df["isComment_bool"]) &
    (~df["isRetweet_bool"]) &
    (~df["isDeleted_bool"]) &
    (~df["isAdvertisement_bool"]) &
    (~df["isBot_bool"]) &
    (df["parentId_clean"].isna())
]

print("Candidatos a root (simple):", len(root_candidates))
print(root_candidates[["id", "threadId", "createdAt", "parentId", "text"]].head())
